In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

import sys
import os
import pickle
from tqdm import tqdm
# Get the absolute path to the folder containing `utils`
utils_path = os.path.abspath('../')
if utils_path not in sys.path:
    sys.path.append(utils_path)

os.environ["CUDA_VISIBLE_DEVICES"] = "1" # second gpu
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"

from utils import *

# az.style.use("arviz-docgrid")
plt.rcParams['figure.dpi'] = 140

experiment_orientations = [159, 123, 87, 51, 15]
subjects = ["01", "02", "03", "04", "05", "06", "07" ,"09", "10", "11", "12"]
median_key = {15:0, 51:1, 87:2, 123:3, 159:4}
std_key = {15:0, 51:1, 87:2, 123:3, 159:4}

c_table = pd.read_csv('../data_caches/ctiraltable.csv')
med = np.load('../data_caches/med.npy')
std = np.load('../data_caches/std.npy')
ctimetable = np.load('../data_caches/ctimetable.npy')
r_table = pd.read_csv('../data_caches/rtrialtable.csv', index_col=0)
(x, y, d, r, e, cd, ce) = np.load('../data_caches/rtimetable.npy', allow_pickle=True)

In [2]:
import jax
jax.config.update('jax_platform_name', 'gpu')

import jax.numpy as jnp
import jax.random as jr
from jax import lax
from jax import vmap
import optax

from jax.extend import backend
print(backend.get_backend().platform)

gpu


In [3]:
def train_hmm(model, emissions, inputs, verbose = True):
    parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior", emissions = emissions)
    fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = emissions, inputs = inputs, num_iters = 10,verbose = verbose)
    return model, fit_params, lps

def scale_to_bounds(x, axis=0, eps=1e-3):
    """
    Linearly rescales x along `axis` so that
      min → eps, max → 1-eps,
    and everything else is in between.
    """
    # compute min and max (keepdims so we can broadcast)
    mn = x.min(axis=axis, keepdims=True)
    mx = x.max(axis=axis, keepdims=True)
    # normalize to [0,1]
    x0 = (x - mn) / (mx - mn)
    # stretch to [eps, 1-eps]
    return x0 * (1 - 2*eps) + eps

## Data

In [4]:
# Modeling setup:
# emissions : 3 components -> reaction time, response angle, response error

# inputs for them:
# reaction time: error, surprise | attention, coherence, expectiation | bias
# response angle: stim, prev. stim, pre. resp | attention, coherence, expectation | bias
# response error: | attention, coherence, expectation | bias

# models for them:
# reaction time: Gamma GLM
# response angle: von Mises GLM
# response error: Beta GLM

In [5]:
# Data Prepping

# copy r_table completely to keep backup
rdf = r_table.copy()

# Add 2 more columns,
# - target at previous trial
# - response at previous trial

rdf['prev_target'] = rdf.groupby(['subj', 'sess', 'run'])['t_calib'].shift(1)
rdf['prev_resp']   = rdf.groupby(['subj', 'sess', 'run'])['resp'].shift(1)

# Replace NaN values in the first trial of each group with the current trial's value
rdf['prev_target'] = rdf['prev_target'].fillna(rdf['t_calib'])
rdf['prev_resp']   = rdf['prev_resp'].fillna(rdf['resp'])


In [ ]:

# Fundamental input parameters
baseline_bias = np.ones((288, 120, 1))

coherence = (rdf.coh == "high").astype(np.float32).values.reshape(-1, 120, 1)
attention = (rdf.att == "focused").astype(np.float32).values.reshape(-1, 120, 1)
expectation = (rdf.exp_al == "expected").astype(np.float32).values.reshape(-1, 120, 1)

# wherever value is 0, set it to -1
coherence = np.where(coherence == 0, -1, coherence)
attention = np.where(attention == 0, -1, attention)
expectation = np.where(expectation == 0, -1, expectation)


resp_err = np.deg2rad(rdf.resp_err.values.reshape(-1, 120, 1))
surprise = np.deg2rad((rdf.t_calib - rdf.exp).values.reshape(-1, 120, 1))
surprise = np.nan_to_num(surprise, nan=0.0)

stim = np.deg2rad(rdf.t_calib.values.reshape(-1, 120, 1))
prev_stim = np.deg2rad(rdf.prev_target.values.reshape(-1, 120, 1))
prev_resp = np.deg2rad(rdf.prev_resp.values.reshape(-1, 120, 1))

# Emissions
# min max scale it using first axis - we are scaling each trial independently
# Reaction time: subtract baseline and convert to ms/frame
rt_raw = (r_table.f_resp.values - 250).reshape(-1, 120, 1) * (1000/120)
reaction_time = scale_to_bounds(rt_raw, axis=0, eps=1e-3)

# Response angle stays as radians (no scaling necessary)
response_angle = np.deg2rad(r_table.resp.values).reshape(-1, 120, 1)

# Response error: convert to radians then rescale into (eps,1-eps)
err_raw = np.deg2rad(r_table.resp_err.values).reshape(-1, 120, 1)
response_error = scale_to_bounds(err_raw, axis=0, eps=1e-3)

In [ ]:

# print and check if any of these have nan in them
print("coherence", np.isnan(coherence).sum())
print("attention", np.isnan(attention).sum())
print("expectation", np.isnan(expectation).sum())

print("surprise", np.isnan(surprise).sum())
print("resp_err", np.isnan(resp_err).sum())

print("stim", np.isnan(stim).sum())
print("prev_stim", np.isnan(prev_stim).sum())
print("prev_resp", np.isnan(prev_resp).sum())

print("reaction_time", np.isnan(reaction_time).sum())
print("response_angle", np.isnan(response_angle).sum())
print("response_error", np.isnan(response_error).sum())

coherance 0
attention 0
expectation 0
surprise 0
resp_err 0
stim 0
prev_stim 0
prev_resp 0
reaction_time 0
response_angle 0
response_error 0


In [ ]:
# lets package emissions and inputs
# reaction time: error, surprise | attention, coherence, expectiation | bias
# response angle: stim, prev. stim, pre. resp | attention, coherence, expectation | bias
# response error: | attention, coherence, expectation | bias

# exactly in this order
inputs = np.concatenate(
    (
        resp_err, surprise, attention, coherence, expectation, baseline_bias,
        stim, prev_stim, prev_resp, attention, coherence, expectation, baseline_bias,
        attention, coherence, expectation, baseline_bias
    ), 
    axis=-1
)

emissions = np.concatenate(
    (
        reaction_time, response_angle, response_error
    ), 
    axis=-1
)

emissions.shape, inputs.shape

((288, 120, 3), (288, 120, 17))

## Global

In [9]:
num_states, input_dim, emission_dim = 2, 17, 3

In [10]:
# # Multi-state model cross-validation
# global_crossval = {}

# for nstate in tqdm(range(2, 10)):
#     model = BlockHMM(nstate, input_dim, emission_dim)
#     # make index of emissions size and shufflei t
#     shuffled_indices = np.random.permutation(emissions.shape[0])
#     parameters, properties = model.initialize(key=jr.PRNGKey(nstate), method="kmeans", emissions = emissions[shuffled_indices])
#     ll_mean, ll = cross_validate_regressor(model=model, emissions=emissions[shuffled_indices], key=jr.PRNGKey(0), num_iters=500, inputs=inputs[shuffled_indices])
#     global_crossval[nstate] = ll

In [11]:
# with open('./caches/global_crossval.pkl', 'wb') as f:
#     pickle.dump(global_crossval, f)
    
# with open('./caches/global_crossval.pkl', 'rb') as f:
#     global_crossval = pickle.load(f)

## Full training the global params

In [13]:
global_params = {}

for nstate in tqdm(range(2, 10)):
    model = BlockHMM(nstate, input_dim, emission_dim)
    parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior", emissions = emissions)
    subset_idx = np.random.choice(np.arange(0, len(emissions)), len(emissions), replace=False)
    fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = emissions[subset_idx], inputs = inputs[subset_idx], num_iters = 500, verbose = False)
    global_params[nstate] = fit_params

 50%|█████     | 4/8 [01:47<01:46, 26.72s/it]E0422 12:00:32.503007  506123 hlo_lexer.cc:443] Failed to parse int literal: 554909908499626281254
E0422 12:00:33.203817  506123 hlo_lexer.cc:443] Failed to parse int literal: 215766023842092697283172
100%|██████████| 8/8 [03:37<00:00, 27.13s/it]


In [ ]:
# plt.figure(figsize=(5, 5))
# plt.plot(lps)

In [14]:
with open('./caches/global_params.pkl', 'wb') as f:
    pickle.dump(global_params, f)
    
with open('./caches/global_params.pkl', 'rb') as f:
    global_params = pickle.load(f)

## Crossvalidate subject models LOO

In [15]:
# One state model cross-validation
# One state model does not need global since there is no state switching
one_crossval_results = {}

for idx, (s_em, s_in) in tqdm(enumerate(zip(emissions.reshape(12, 4 * 6, 120, 3), inputs.reshape(12, 4 * 6, 120, 17)))):
    one_crossval_results[idx] = {}
    shuffle_idx = np.random.permutation(len(s_em))
    model = BlockOneRegressor()
    ll_mean, ll = cross_validate_regressor(model=model, emissions=s_em[shuffle_idx], key=jr.PRNGKey(0), num_iters=5000, inputs=s_in[shuffle_idx], init="default")
    one_crossval_results[idx][1] = ll

12it [05:02, 25.17s/it]


In [16]:
with open('./caches/one_state_crossval_result.pkl', 'wb') as f:
    pickle.dump(one_crossval_results, f)
    
with open('./caches/one_state_crossval_result.pkl', 'rb') as f:
    one_crossval_results = pickle.load(f)

In [17]:
crossval_results = {}

for idx, (s_em, s_in) in enumerate(zip(emissions.reshape(12, 4 * 6, 120, 3), inputs.reshape(12, 4 * 6, 120, 17))):
    crossval_results[idx] = {}
    shuffle_idx = np.random.permutation(len(s_em))
    
    for nstate in tqdm(range(2, 10), desc=f'sub-{idx+1}'):
        model = BlockHMM(nstate, input_dim, emission_dim)
        gpar = global_params[nstate]
        parameters, properties = model.initialize(key=jr.PRNGKey(1), 
                                                  method="prior",
                                                  initial_probs=gpar.initial.probs,
                                                  transition_matrix=gpar.transitions.transition_matrix,
                                                  weights_rt = gpar.emissions.weights_rt,
                                                  alpha_rt = gpar.emissions.alpha_rt,
                                                  weights_ra = gpar.emissions.weights_ra,
                                                  kappa_ra = gpar.emissions.kappa_ra,
                                                  weights_re = gpar.emissions.weights_re,
                                                  phi_re = gpar.emissions.phi_re)
        ll_mean, ll = cross_validate_regressor(model=model, emissions=s_em[shuffle_idx], key=jr.PRNGKey(0), num_iters=500, inputs=s_in[shuffle_idx], init = (parameters, properties))
        crossval_results[idx][nstate] = ll

sub-12: 100%|██████████| 8/8 [03:35<00:00, 26.98s/it]


In [18]:
with open('./caches/sub_crossval_results.pkl', 'wb') as f:
    pickle.dump(crossval_results, f)
with open('./caches/sub_crossval_results.pkl', 'rb') as f:
    crossval_results = pickle.load(f)

## Subject params full training

In [19]:
subject_params = {}

for idx, (s_em, s_in) in enumerate(zip(emissions.reshape(12, 4 * 6, 120, 3), inputs.reshape(12, 4 * 6, 120, 17))):
    subject_params[idx] = {}
    shuffle_idx = np.random.permutation(len(s_em))
    
    nstate = 2
    model = BlockHMM(nstate, input_dim, emission_dim)
    gpar = global_params[nstate]
    parameters, properties = model.initialize(key=jr.PRNGKey(1), 
                                                method="prior",
                                                initial_probs=gpar.initial.probs,
                                                transition_matrix=gpar.transitions.transition_matrix,
                                                weights_rt = gpar.emissions.weights_rt,
                                                alpha_rt = gpar.emissions.alpha_rt,
                                                weights_ra = gpar.emissions.weights_ra,
                                                kappa_ra = gpar.emissions.kappa_ra,
                                                weights_re = gpar.emissions.weights_re,
                                                phi_re = gpar.emissions.phi_re)
    fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = s_em[shuffle_idx], inputs = s_in[shuffle_idx], num_iters = 500, verbose = False)
    subject_params[idx][nstate] = fit_params

In [20]:
with open('./caches/subject_params.pkl', 'wb') as f:
    pickle.dump(subject_params, f)
    
with open('./caches/subject_params.pkl', 'rb') as f:
    subject_params = pickle.load(f)

## Most likely states

In [21]:
ml_states = {}
nstate = 2

cem = emissions.reshape(12, 4 * 6, 120, 3)
cin = inputs.reshape(12, 4 * 6, 120, 17)
for idx in tqdm(range(12), desc=f''):
    shuffle_idx = np.random.permutation(len(cem[idx]))
    
    em = jnp.array(cem[idx].reshape(-1, 120, 3)[shuffle_idx])
    inps = jnp.array(cin[idx].reshape(-1, 120, 17)[shuffle_idx])
    
    model = BlockHMM(nstate, input_dim, emission_dim)
    
    gpar = subject_params[idx][nstate]
    parameters, properties = model.initialize(key=jr.PRNGKey(1), 
                                                method="prior",
                                                initial_probs=gpar.initial.probs,
                                                transition_matrix=gpar.transitions.transition_matrix,
                                                weights_rt = gpar.emissions.weights_rt,
                                                alpha_rt = gpar.emissions.alpha_rt,
                                                weights_ra = gpar.emissions.weights_ra,
                                                kappa_ra = gpar.emissions.kappa_ra,
                                                weights_re = gpar.emissions.weights_re,
                                                phi_re = gpar.emissions.phi_re)
    # fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = em[shuffle_idx], inputs = inps[shuffle_idx], num_iters = 100, verbose = False)
    # Define a vmapped version of most_likely_states.
    # This applies the function to each trial in the batch dimension of `em`.
    most_likely_states_vmap = vmap(lambda trial, inp: model.most_likely_states(parameters, trial, inp))
    t_trail = most_likely_states_vmap(em, inps)
    ml_states[idx] = t_trail

100%|██████████| 12/12 [00:07<00:00,  1.62it/s]


In [23]:
with open('./caches/ml_states.pkl', 'wb') as f:
    pickle.dump(ml_states, f)
    
with open('./caches/ml_states.pkl', 'rb') as f:
    ml_states = pickle.load(f)